In [ ]:
!pip install fastapi uvicorn transformers torch

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load Hugging Face model and tokenizer (e.g., CodeGen or CodeT5)
model_name = "Salesforce/codegen-350M-mono"  # You can change to other models like CodeT5, CodeBERT, etc.
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

app = FastAPI()

# Define request and response data models
class CodeRequest(BaseModel):
    code: str

class PromptRequest(BaseModel):
    prompt: str

class CodeResponse(BaseModel):
    response: str

# Helper function to call Hugging Face models
def generate_code_from_model(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model.generate(**inputs, max_length=512, num_beams=5, temperature=0.7, no_repeat_ngram_size=2)
    generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_code

# Endpoint to generate code from a natural language prompt
@app.post("/generate_code", response_model=CodeResponse)
async def generate_code(request: PromptRequest):
    prompt = f"Write code for: {request.prompt}"
    code = generate_code_from_model(prompt)
    return CodeResponse(response=code)

# Endpoint to debug code
@app.post("/debug_code", response_model=CodeResponse)
async def debug_code(request: CodeRequest):
    code = f"Debug the following code:\n{request.code}"
    debugged_code = generate_code_from_model(code)
    return CodeResponse(response=debugged_code)

# Endpoint to optimize code
@app.post("/optimize_code", response_model=CodeResponse)
async def optimize_code(request: CodeRequest):
    code = f"Optimize the following code:\n{request.code}"
    optimized_code = generate_code_from_model(code)
    return CodeResponse(response=optimized_code)

# Endpoint to write unit tests for code
@app.post("/write_tests", response_model=CodeResponse)
async def write_tests(request: CodeRequest):
    tests = f"Write unit tests for the following code:\n{request.code}"
    test_code = generate_code_from_model(tests)
    return CodeResponse(response=test_code)

# Start FastAPI server (run with 'uvicorn main:app --reload')
